# Multifractal spectra of **real, experimentally synthesised** MOFs

This notebook is the twin of `mof_band_analysis.ipynb`. Every analysis cell is
identical — same bonding rule, same supercell rule, same influential-node
selection, same τ definition, same q grid. Only the **data source** differs.

## Why this notebook exists

The 77-framework band was computed on **hMOF** structures: chemically sound, but
generated by a program and never made in a laboratory. That happened by
accident. The original notebook asked MOFX-DB for `CoREMOF 2019` and received
hMOF records instead, because on that endpoint the `database=` parameter is
advisory and loses to the `gases[]` filter when both are supplied. The request
was right; the response did not honour it.

A provenance check further down the original notebook caught it and printed a
warning. That warning is the only reason the project did not publish
"experimental" results computed on generated structures.

## What changed

Step 2 now **filters every returned record on its own `database` field** and
discards anything that is not CoRE MOF, rather than trusting the server. It also
prints how many records were streamed, how many were kept, and what was thrown
away — so the provenance claim is checkable rather than asserted.

If the live query fails or the service is unreachable, Step 2 falls back to
reading CIF files from a local folder, so this notebook still runs.

## What to do with the output

The final cell writes **`real_results.json`** and a set of figures. Send
`real_results.json` back and it goes straight onto the Results page of the
project site alongside the 77-framework band.

**Runtime:** roughly 15–25 minutes for 50 frameworks on a free Colab CPU
instance. The expensive part is the all-pairs shortest-path computation on each
supercell, which is `O(N²)` in memory and `O(N·E)` in time.


## Step 1 — Setup

In [ ]:
# ============================================================
# STEP 1: Setup
# ============================================================
!pip install -q ase networkx scipy matplotlib requests 2>/dev/null || pip install -q --break-system-packages ase networkx scipy matplotlib requests

import io, json, time, warnings
from pathlib import Path

import numpy as np
import requests
import networkx as nx
import matplotlib.pyplot as plt
from ase.io import read
from ase.neighborlist import neighbor_list
from scipy.sparse.csgraph import shortest_path

warnings.filterwarnings("ignore")

OUT_DIR = Path("multi_mof_output"); OUT_DIR.mkdir(exist_ok=True)

MOFDB_BASE = "https://mof.tech.northwestern.edu"
TARGET_DATABASE = "CoREMOF 2019"   # real, experimentally-derived crystal structures
TARGET_GAS      = "Xenon"          # server-side filter: must have a simulated Xe isotherm
MIN_MOFS        = 50               # the ask: at least 50 frameworks

# ---- analysis parameters (identical to before; all tunable, stability checked in Step 8)
MIN_CELL_LENGTH  = 48.0   # A -- supercell must reach this in every lattice direction
MAX_ATOMS        = 7000   # all-pairs distance matrix is O(N^2) in memory
INFLUENTIAL_FRAC = 0.10
Q_VALUES         = np.linspace(-10, 10, 41)
WINDOW_FRAC      = 0.34
N_BATCHES, TRIALS_PER_BATCH = 6, 6

METALS = {"Zn","Cu","Ni","Co","Fe","Al","Cr","Mn","Zr","Ti","Mg","Ca","V","Cd","Ag",
          "Sc","Y","La","Ce","Nd","Gd","Dy","Er","Yb","In","Ga","Sn","Pb","Mo","W","Nb","Ta","Hf"}


## Step 2 — Query MOFX-DB for real CoRE MOF structures, and **verify** what comes back

The fix is the `_is_target_db()` filter: each record is checked against the
database it says it came from, and anything that is not CoRE MOF is discarded
before it can reach the analysis.


In [ ]:
# ============================================================
# STEP 2: Live query against MOFX-DB — with a hard client-side provenance filter
# ============================================================
# THE BUG THIS FIXES
# The previous run asked for database="CoREMOF 2019" and got hMOF records back.
# On this endpoint the database= parameter is advisory: when it is combined with
# a gases[] filter, the gas filter wins and the database filter is ignored. The
# request was correct and the response was wrong, which is the worst combination
# because nothing errors -- you just silently analyse the wrong material class.
#
# THE FIX
# Trust nothing the server says about what it sent. Check every record against
# its own `database` field and throw away anything that does not match. This is
# the difference between "we asked for real MOFs" and "these are real MOFs".
!pip install -q mofdb-client 2>/dev/null || pip install -q --break-system-packages mofdb-client

import collections, re
from pathlib import Path

USE_LIVE_QUERY = True     # set False to skip the API and use LOCAL_CIF_DIR below
LOCAL_CIF_DIR  = Path("real_cifs")   # fallback: a folder of .cif files you supply


def _norm(s):
    """Loose comparison: 'CoREMOF 2019', 'CoRE-MOF 2019' and 'coremof2019' all match."""
    return str(s or "").lower().replace(" ", "").replace("-", "").replace("_", "")


def _is_target_db(m):
    """True only if THIS record actually comes from the database we asked for."""
    return _norm(TARGET_DATABASE) in _norm(getattr(m, "database", ""))


def fetch_real_mofs(want, gas, hard_cap=6000):
    """Stream MOFX-DB and keep only genuine CoRE MOF records.

    want      how many verified records we need before stopping
    gas        adsorbate to filter on, or None for no gas filter at all
    hard_cap  stop scanning if the server is clearly ignoring the filter
              entirely, rather than reading a very long stream to the end
    """
    from mofdb_client.main import get_all
    params = {"database": TARGET_DATABASE}
    if gas:
        params["gases[]"] = [gas]
    kept, seen, rejected = [], 0, collections.Counter()
    for m in get_all(params):
        seen += 1
        if _is_target_db(m):
            kept.append(m)
        else:
            rejected[str(getattr(m, "database", "unknown"))] += 1
        if len(kept) >= want or seen >= hard_cap:
            break
    label = f"with gas={gas}" if gas else "with NO gas filter"
    print(f"  [{label}] streamed {seen}; kept {len(kept)} verified {TARGET_DATABASE}")
    if rejected:
        print(f"    discarded (wrong database): {dict(rejected.most_common(5))}")
    return kept


candidates = []
if USE_LIVE_QUERY:
    try:
        # ATTEMPT 1: with the gas filter, so the set stays Xe-relevant.
        candidates = fetch_real_mofs(want=max(80, MIN_MOFS * 2), gas=TARGET_GAS)

        # ATTEMPT 2: without it. This is not just a fallback -- it is the more
        # likely path to work. The whole bug was that `database=` loses to
        # `gases[]` when both are sent, so removing the gas filter should let
        # the server honour the database filter properly. The cost is that the
        # structures are no longer guaranteed to have a simulated Xe isotherm,
        # which this analysis does not use anyway: nothing downstream reads the
        # isotherms, only the crystal structure.
        if len(candidates) < MIN_MOFS:
            print(f"\n  only {len(candidates)} survived with the gas filter; "
                  f"retrying without it")
            candidates = fetch_real_mofs(want=max(80, MIN_MOFS * 2), gas=None)
    except Exception as e:
        print(f"  live query failed ({type(e).__name__}: {e})")
        print(f"  falling back to local CIFs in {LOCAL_CIF_DIR}/")

# ---- fallback: a folder of CIF files ---------------------------------------
# Used if the API is unreachable, or if you would rather supply a curated set.
# Put .cif files in real_cifs/ (upload a zip and unzip it) and re-run this cell.
if not candidates:
    class _LocalMof:
        """Minimal stand-in with the same attributes Step 3 reads off a record."""
        def __init__(self, path, idx):
            self.cif = path.read_text()
            self.name = path.stem
            self.id = idx
            self.database = TARGET_DATABASE + " (local file)"
            self.mofid = self.mofkey = None
            self.url = str(path)
            self.void_fraction = self.lcd = self.pld = None
            self.isotherms = []
    if LOCAL_CIF_DIR.exists():
        files = sorted(LOCAL_CIF_DIR.glob("*.cif"))
        candidates = [_LocalMof(p, i) for i, p in enumerate(files)]
        print(f"  loaded {len(candidates)} CIFs from {LOCAL_CIF_DIR}/")
    else:
        LOCAL_CIF_DIR.mkdir(exist_ok=True)
        print(f"  created {LOCAL_CIF_DIR}/ — put .cif files there and re-run this cell")

candidates = sorted(candidates, key=lambda m: m.id)

# ---- VERIFY WHAT WE ACTUALLY HAVE ------------------------------------------
returned_dbs = collections.Counter(str(getattr(m, "database", "unknown")) for m in candidates)
print()
print(f"{len(candidates)} structures ready.")
print(f"  databases represented: {dict(returned_dbs)}")

_wrong = [db for db in returned_dbs if not _norm(TARGET_DATABASE) in _norm(db)]
assert not _wrong, (
    f"Provenance check FAILED: {_wrong} slipped through the filter. "
    f"Do not analyse this set -- fix the filter first.")
print("  provenance: every record verified as", TARGET_DATABASE)

# ---- metal diversity: the thing the hMOF set completely lacked --------------
_metals = collections.Counter()
for m in candidates:
    els = set(re.findall(r"\[([A-Z][a-z]?)[\]\[]", getattr(m, "mofid", "") or ""))
    els -= {"C", "N", "O", "H", "S", "P", "Cl", "F", "Br", "I", "Si"}
    _metals[",".join(sorted(els)) or "unparsed"] += 1
print(f"  metal composition: {dict(_metals.most_common(10))}")
if len([k for k in _metals if k != "unparsed"]) <= 2:
    print("  *** NOTE: this set is effectively single-metal. Do not claim it")
    print("  *** 'cuts across different metals' -- the hMOF set failed on exactly this.")
else:
    print("  -> multi-metal set: a cross-family claim is testable on this data")

assert len(candidates) >= MIN_MOFS, (
    f"Only {len(candidates)} verified structures, below MIN_MOFS={MIN_MOFS}.\n"
    f"  -> If this is 0, MOFX-DB returned nothing. Use STEP 2b (Route B) below:\n"
    f"     set RUN_ROUTE_B = True and run it, then continue from Step 3.\n"
    f"     It pulls CoRE MOF 2019 from its permanent Zenodo DOI instead.\n"
    f"  -> Otherwise: raise `want`, set TARGET_GAS = None, or put CIFs in\n"
    f"     {LOCAL_CIF_DIR}/ and set USE_LIVE_QUERY = False.\n"
    f"  Do NOT pad this with hMOF entries.")

for m in candidates[:5]:
    print(f"  id={m.id:>7}  name={str(m.name)[:26]:<26}  db={m.database}")
print("  ...")


### Step 2b — Route B: CoRE MOF straight from Zenodo (no MOFX-DB)

**Run this only if Step 2 returned zero records.**

MOFX-DB is a live service and can be unreachable, rate-limited, or changed. The
underlying dataset does not depend on it: CoRE MOF 2019 is archived on Zenodo
under a permanent DOI, which is a more reliable source anyway — a fixed archive
cannot silently return a different database the way the API did.

This cell downloads the archive, samples structures **stratified by metal** so
the set is not accidentally single-metal like the hMOF one, and hands them to
the same Step 3 that the API route feeds. Everything downstream is unchanged.


In [ ]:
# ============================================================
# STEP 2b: CoRE MOF 2019 from the Zenodo archive — the no-API route
# ============================================================
# WHY THIS IS THE BETTER SOURCE, not just a fallback:
# A live API can return whatever it likes and did (we asked for CoRE MOF and
# got hMOF). A Zenodo record is immutable and citable — DOI 10.5281/zenodo.3370236
# is the same bytes for everyone, forever. If provenance matters, and here it is
# the entire point, a fixed archive beats a query.
#
# Set RUN_ROUTE_B = True and run this cell, then continue from Step 3.

RUN_ROUTE_B = False          # <-- flip to True to use this route

CORE_MOF_DOI = "10.5281/zenodo.3370236"   # CoRE MOF 2019, public release
N_WANTED     = 60                          # structures to sample
PER_METAL    = 12                          # cap per metal, so no single metal dominates

if RUN_ROUTE_B:
    import re, shutil, random, collections
    from pathlib import Path

    DL = Path("core_mof_download"); DL.mkdir(exist_ok=True)

    # zenodo_get resolves a DOI to its files and downloads them, so we do not
    # have to hardcode a filename that may change between record versions.
    if not any(DL.iterdir()):
        !pip install -q zenodo_get
        !zenodo_get {CORE_MOF_DOI} -o {DL}
    print("downloaded:", [p.name for p in DL.iterdir()])

    # ---- unpack whatever archive came down --------------------------------
    CIF_POOL = Path("core_mof_cifs"); CIF_POOL.mkdir(exist_ok=True)
    if not any(CIF_POOL.glob("*.cif")):
        for arc in DL.iterdir():
            if arc.suffix.lower() in (".zip", ".gz", ".tar", ".tgz", ".xz"):
                print(f"unpacking {arc.name} ...")
                try:
                    shutil.unpack_archive(str(arc), str(CIF_POOL))
                except Exception as e:
                    print(f"  could not unpack ({e}); skipping")
    # CIFs may sit in nested folders; flatten the ones we find.
    found = list(CIF_POOL.rglob("*.cif"))
    print(f"{len(found)} CIF files available in the archive")
    assert found, ("No .cif files found. Open core_mof_cifs/ in the file browser "
                   "and check what the archive actually contained.")

    # ---- sample, stratified by metal --------------------------------------
    # The hMOF set was accidentally single-metal, which made every cross-family
    # question unanswerable. Capping per metal here stops that repeating.
    METALS = {"Zn","Cu","Ni","Co","Fe","Al","Cr","Mn","Zr","Ti","Mg","Ca","V","Cd",
              "Ag","Sc","Y","La","Ce","Nd","Gd","Dy","Er","Yb","In","Ga","Sn","Pb",
              "Mo","W","Nb","Ta","Hf"}

    def metals_in(cif_path):
        """Read only the atom_site block — full parsing here would be wasteful."""
        try:
            txt = cif_path.read_text(errors="ignore")
        except Exception:
            return set()
        syms = set(re.findall(r"^\s*([A-Z][a-z]?)\d*\s", txt, re.M))
        return syms & METALS

    random.seed(0)                      # reproducible sample
    random.shuffle(found)
    picked, by_metal = [], collections.Counter()
    for p in found:
        ms = metals_in(p)
        if not ms:
            continue
        key = ",".join(sorted(ms))
        if by_metal[key] >= PER_METAL:
            continue
        by_metal[key] += 1
        picked.append(p)
        if len(picked) >= N_WANTED:
            break

    LOCAL_CIF_DIR = Path("real_cifs"); LOCAL_CIF_DIR.mkdir(exist_ok=True)
    for p in picked:
        shutil.copy(p, LOCAL_CIF_DIR / p.name)

    print(f"\nsampled {len(picked)} structures into {LOCAL_CIF_DIR}/")
    print(f"metal composition: {dict(by_metal.most_common(12))}")
    if len(by_metal) <= 2:
        print("  *** still effectively single-metal — widen PER_METAL or N_WANTED")
    else:
        print(f"  -> {len(by_metal)} distinct metal compositions: a cross-family claim is testable")

    # ---- hand them to Step 3 in the shape it expects ----------------------
    class _LocalMof:
        """Same attribute surface Step 3 reads off an API record."""
        def __init__(self, path, idx):
            self.cif = path.read_text(errors="ignore")
            self.name = path.stem
            self.id = idx
            self.database = "CoREMOF 2019 (Zenodo " + CORE_MOF_DOI + ")"
            self.mofid = self.mofkey = None
            self.url = "https://doi.org/" + CORE_MOF_DOI
            self.void_fraction = self.lcd = self.pld = None
            self.isotherms = []

    candidates = [_LocalMof(p, i) for i, p in enumerate(sorted(LOCAL_CIF_DIR.glob("*.cif")))]
    print(f"\ncandidates ready: {len(candidates)} structures")
    print("Provenance: CoRE MOF 2019, experimentally derived, all solvent removed.")
    print("Continue from Step 3.")
else:
    print("Route B is off. Set RUN_ROUTE_B = True above if Step 2 returned nothing.")


## Step 3 — Parse each CIF, keep provenance, don't hide failures

Real experimental structures are messier than a hand-picked set of 8 textbook frameworks:
disordered sites, missing symmetry info, or a CIF ASE simply can't parse are all expected at
this scale. Every failure is caught **individually, logged with its reason, and excluded** —
never silently. This is different from the previous notebook's "no bare except" stance, which
was appropriate for 8 CIFs already known to parse cleanly; at 50+ live-fetched real structures,
catching and reporting per-structure failures *is* the honest behavior, not a way to hide bugs.
The full skip list is printed at the end of this step, not swept away.

In [ ]:
# ============================================================
# STEP 3: Parse CIFs (embedded in each Mof object's .cif field, not re-downloaded)
# ============================================================
CIF_DIR = Path("mof_cifs"); CIF_DIR.mkdir(exist_ok=True)

structures = {}
skipped = []

for m in candidates:
    name = m.name
    key = f"{name} (id={m.id})" if name in structures else name
    try:
        cif_text = m.cif
        if not cif_text or not cif_text.strip():
            raise ValueError("empty CIF field in database record")
        path = CIF_DIR / f"{m.id}_{name}.cif"
        path.write_text(cif_text)
        atoms = read(path)
        if not atoms.pbc.all():
            raise ValueError("CIF did not yield a periodic cell")
        if len(atoms) < 4:
            raise ValueError(f"only {len(atoms)} atoms after parsing -- too small to be real")
        structures[key] = dict(
            atoms=atoms, mof_id=m.id, mofid=m.mofid, mofkey=m.mofkey,
            database=m.database, url=m.url,
            void_fraction=m.void_fraction, lcd=m.lcd, pld=m.pld,
            n_xe_isotherms=sum(1 for iso in m.isotherms
                                if any(TARGET_GAS.lower() in str(a).lower()
                                       for a in getattr(iso, "adsorbates", []))),
        )
    except Exception as e:
        skipped.append((name, str(e)))

print(f"Parsed {len(structures)} / {len(candidates)} structures successfully.")
if skipped:
    print(f"\nSkipped {len(skipped)} (reason logged, not hidden):")
    for n, reason in skipped:
        print(f"  - {n}: {reason}")

assert len(structures) >= MIN_MOFS, (
    f"Only {len(structures)} structures parsed cleanly, below MIN_MOFS={MIN_MOFS}. "
    f"Widen the candidate pool (raise limit in Step 2) rather than lowering this bar.")


## Step 4 — Bonding, with periodic boundary conditions (identical method to before)

In [ ]:
# ============================================================
# STEP 4: PBC-aware bond graph
# ============================================================
def bond_cutoff(e1, e2):
    pair = {e1, e2}
    m1, m2 = e1 in METALS, e2 in METALS
    if m1 and m2:          return 2.80
    if m1 or m2:           return 2.45
    if pair <= {"C","O","N"}: return 1.70
    if "H" in pair or "F" in pair: return 1.45
    return 1.90

def cutoff_dict(atoms):
    els = sorted(set(atoms.get_chemical_symbols()))
    return {(a, b): bond_cutoff(a, b) for a in els for b in els}

def build_graph(atoms, pbc=True):
    at = atoms if pbc else atoms.copy()
    if not pbc:
        at.set_pbc(False)
    i, j = neighbor_list("ij", at, cutoff_dict(at))
    G = nx.Graph()
    G.add_nodes_from(range(len(at)))
    G.add_edges_from((int(a), int(b)) for a, b in zip(i, j) if a < b)
    return G

disconnected = []
for name in list(structures.keys()):
    s = structures[name]
    G = build_graph(s["atoms"], pbc=True)
    if not nx.is_connected(G):
        disconnected.append(name)
        continue
    s["G_unit"] = G

for name in disconnected:
    print(f"Dropping '{name}': framework graph not connected under PBC "
          f"(likely an interpenetrated or guest-only fragment issue).")
    del structures[name]

print(f"\n{len(structures)} structures have a connected PBC bond graph.")
assert len(structures) >= MIN_MOFS, f"Only {len(structures)} left after connectivity filtering."


## Step 5 — Supercell and coarse-graining (identical method to before)

In [ ]:
# ============================================================
# STEP 5: Supercell + coarse-graining
# ============================================================
def make_supercell(atoms, min_length=MIN_CELL_LENGTH, max_atoms=MAX_ATOMS):
    L = atoms.cell.lengths()
    rep = [int(max(1, np.ceil(min_length / l))) for l in L]
    while len(atoms) * int(np.prod(rep)) > max_atoms and max(rep) > 1:
        k = int(np.argmax(L * np.array(rep)))
        if rep[k] == 1:
            k = int(np.argmax(rep))
        rep[k] -= 1
    return atoms.repeat(tuple(rep)), tuple(rep)

def coarse_grain(G, sym):
    metals = [n for n in G if sym[n] in METALS]
    node_atoms = set(metals)
    for m in metals:
        node_atoms.update(o for o in G[m] if sym[o] == "O")
    nodes   = [sorted(c) for c in nx.connected_components(G.subgraph(node_atoms))]
    organic = [n for n in G if n not in node_atoms]
    linkers = [sorted(c) for c in nx.connected_components(G.subgraph(organic))]
    return nodes, linkers

failed_supercell = []
for name in list(structures.keys()):
    s = structures[name]
    try:
        sc, rep = make_supercell(s["atoms"])
        G = build_graph(sc, pbc=True)
        if not nx.is_connected(G):
            raise ValueError("supercell framework disconnected")
        sym = np.array(sc.get_chemical_symbols())
        nodes, linkers = coarse_grain(G, sym)
        s.update(super_atoms=sc, rep=rep, G=G, sym=sym, nodes=nodes, linkers=linkers)
    except Exception as e:
        failed_supercell.append((name, str(e)))
        del structures[name]

if failed_supercell:
    print(f"Dropped {len(failed_supercell)} at the supercell stage:")
    for n, reason in failed_supercell:
        print(f"  - {n}: {reason}")

print(f"\n{len(structures)} structures ready for iNMFA.")
assert len(structures) >= MIN_MOFS, f"Only {len(structures)} left after supercell stage."


## Step 6 — iNMFA: influential-node multifractal analysis (identical method to before)

In [ ]:
# ============================================================
# STEP 6: iNMFA implementation (unchanged from the previous notebook)
# ============================================================
INT16_MAX = np.iinfo(np.int16).max

def dist_matrix(G):
    A = nx.to_scipy_sparse_array(G, nodelist=sorted(G.nodes()), format="csr")
    D = shortest_path(A, method="D", unweighted=True, directed=False)
    D[~np.isfinite(D)] = INT16_MAX
    return D.astype(np.int16)

def box_cover(D, rb, rng):
    n = D.shape[0]
    uncovered = np.ones(n, bool)
    assign = np.full(n, -1)
    sizes, bid = [], 0
    for c in rng.permutation(n):
        if not uncovered[c]:
            continue
        members = uncovered & (D[c] <= rb)
        assign[members] = bid
        sizes.append(int(members.sum()))
        uncovered &= ~members
        bid += 1
        if not uncovered.any():
            break
    return assign, np.array(sizes)

def influential_nodes(G, D, finite, fraction=INFLUENTIAL_FRAC):
    n = G.number_of_nodes()
    deg = np.array([d for _, d in sorted(G.degree())])
    with np.errstate(divide="ignore", invalid="ignore"):
        closeness = (n - 1) / np.where(finite, D, 0).sum(axis=1, dtype=np.int64)
    k = max(2, int(np.ceil(fraction * n)))
    return np.sort(np.lexsort((-closeness, -deg))[:k])

def inmfa(G, q_values=Q_VALUES, fraction=INFLUENTIAL_FRAC, window_frac=WINDOW_FRAC,
          n_batches=N_BATCHES, trials_per_batch=TRIALS_PER_BATCH, seed=0,
          average_pr_first=False, tau_mode="paper"):
    """iNMFA spectrum.

    tau_mode selects how the mass exponent is extracted from ln P_q vs ln(r/r_N):

      "paper"  tau(q) = ln P_q(r) / ln(r/r_N), exactly as written in the source
               paper. Over several radii this is least squares THROUGH THE
               ORIGIN, because P_q ~ (r/r_N)^tau carries no prefactor.

      "slope"  the free-intercept least-squares slope. This is what an earlier
               version of this notebook used, and it is wrong for this formula.

    WHY IT MATTERS -- at q = 0 every term is pr_i^0 = 1, so P_0 = |I|, the count
    of influential nodes, which is the SAME at every radius. A flat line has zero
    slope, so "slope" forces tau(0) = 0 and hence

        f(alpha_0) = 0*alpha - 0 = 0

    But f(alpha_0) is meant to be D_0, the fractal dimension -- the PEAK of the
    spectrum. Pinning it to zero removes the peak, and f(alpha) comes out as a
    falling curve instead of the inverted parabola a multifractal spectrum must
    have. With "paper", ln(r/r_N) < 0 so tau(0) = ln|I| / negative < 0, giving
    f(alpha_0) = -tau(0) > 0 and restoring the peak at q = 0.
    """
    n = G.number_of_nodes()
    D = dist_matrix(G)
    finite = D < INT16_MAX
    diam = int(D[finite].max())
    infl = influential_nodes(G, D, finite, fraction)

    rmax = max(3, int(round(diam * window_frac)))
    radii = list(range(1, rmax + 1))
    x = np.log(np.array(radii, float) / diam)

    tau_b, r2_b = [], []
    for b in range(n_batches):
        if average_pr_first:
            pr_mean = {r: np.zeros(len(infl)) for r in radii}
            for t in range(trials_per_batch):
                rng = np.random.default_rng(seed + 1000 * b + t)
                for r in radii:
                    a, s = box_cover(D, r, rng)
                    pr_mean[r] += s[a[infl]] / n / trials_per_batch
            lnP = np.array([[np.log(np.sum(pr_mean[r][pr_mean[r] > 0] ** q))
                             for r in radii] for q in q_values])
        else:
            lnP = np.zeros((len(q_values), len(radii)))
            for t in range(trials_per_batch):
                rng = np.random.default_rng(seed + 1000 * b + t)
                for ri, r in enumerate(radii):
                    a, s = box_cover(D, r, rng)
                    pr = s[a[infl]] / n
                    pr = pr[pr > 0]
                    lnP[:, ri] += np.log([np.sum(pr ** q) for q in q_values])
            lnP /= trials_per_batch

        tau, r2 = [], []
        for qi, q in enumerate(q_values):
            y = lnP[qi]
            if tau_mode == "paper":
                # tau = ln P_q / ln(r/r_N)  ->  least squares through the origin
                t = float(np.sum(x * y) / np.sum(x * x))
                ss = np.sum((y - y.mean()) ** 2)
                r2.append(np.nan if ss < 1e-12 else 1 - np.sum((y - t * x) ** 2) / ss)
            else:
                t, icept = np.polyfit(x, y, 1)
                ss = np.sum((y - y.mean()) ** 2)
                r2.append(np.nan if (abs(q) < 1e-9 or ss < 1e-12)
                          else 1 - np.sum((y - (t * x + icept)) ** 2) / ss)
            tau.append(t)
        tau_b.append(tau); r2_b.append(r2)

    tau_b = np.array(tau_b)
    alpha_b = np.array([np.gradient(t, q_values) for t in tau_b])
    f_b = q_values[None, :] * alpha_b - tau_b
    i0 = int(np.argmin(np.abs(q_values)))

    def asym(al, fv):
        # alpha_0 is the alpha where f(alpha) is maximum (the paper's definition).
        # With tau_mode="paper" that maximum sits at q = 0, as it should.
        a0 = al[int(np.nanargmax(fv))]
        lo, hi = al.min(), al.max()
        return np.log((a0 - lo) / (hi - a0)) if lo < a0 < hi else np.nan

    widths = alpha_b.max(1) - alpha_b.min(1)
    asyms = np.array([asym(al, fv) for al, fv in zip(alpha_b, f_b)])
    return dict(N=n, diameter=diam, n_influential=len(infl), radii=radii, q=q_values,
                tau=tau_b.mean(0), alpha=alpha_b.mean(0), f_alpha=f_b.mean(0),
                alpha_sd=alpha_b.std(0), f_sd=f_b.std(0),
                r2=np.nanmean(np.array(r2_b), 0),
                width=float(widths.mean()), width_sd=float(widths.std()),
                asymmetry=float(np.nanmean(asyms)), asymmetry_sd=float(np.nanstd(asyms)))


## Step 7 — Run every framework

Per-structure `try/except` here is deliberate (see Step 3's note): at 50+ live-fetched real
structures some will legitimately fail (disconnected influential-node set, degenerate
diameter, numerical edge cases at extreme q). Every failure is printed with its reason and
excluded — the success/fail counts below are the actual numbers, not silently rounded up.

In [ ]:
# ============================================================
# STEP 7: Run
# ============================================================
results = {}
run_failed = []
t_start = time.time()
for name, s in structures.items():
    t0 = time.time()
    try:
        res = inmfa(s["G"])
        res.update(rep=s["rep"], n_nodes=len(s["nodes"]), n_linkers=len(s["linkers"]),
                   mofid=s.get("mofid"), url=s.get("url"), lcd=s.get("lcd"), pld=s.get("pld"))
        results[name] = res
        print(f"{name:28s} N={res['N']:5d} diam={res['diameter']:3d} "
              f"infl={res['n_influential']:5d} fit_pts={len(res['radii']):2d} "
              f"dAlpha={res['width']:.3f}+/-{res['width_sd']:.3f} "
              f"A={res['asymmetry']:+.2f}+/-{res['asymmetry_sd']:.2f} "
              f"minR2={np.nanmin(res['r2']):.3f}  [{time.time()-t0:.0f}s]")
    except Exception as e:
        run_failed.append((name, str(e)))
        print(f"{name:28s} FAILED: {e}")

print(f"\n{len(results)}/{len(structures)} frameworks completed in {time.time()-t_start:.0f}s")
assert len(results) >= MIN_MOFS, (
    f"Only {len(results)} frameworks produced a spectrum, below MIN_MOFS={MIN_MOFS}.")


## Step 6b — save the spectra now

The spectra are expensive to compute and were previously only written to disk inside
the plotting cell, so anything that went wrong afterwards lost the whole run. This
writes `multi_mof_output/results.json` as soon as the numbers exist.


In [ ]:
# ============================================================
# SAVE IMMEDIATELY -- do not wait for the plotting cell
# ============================================================
# The spectra take ~10 minutes to compute. Originally they were only written to
# disk inside the Step 8 plotting cell, so any failure after this point -- a
# matplotlib error, a stopped kernel, closing the tab -- threw all of it away.
# Saving here means the expensive part is durable the moment it exists.
import json
from pathlib import Path
import numpy as np

OUT_DIR = Path("multi_mof_output"); OUT_DIR.mkdir(exist_ok=True)

def _jsonable(v):
    if isinstance(v, np.ndarray):  return v.tolist()
    if isinstance(v, (np.floating, np.integer)): return v.item()
    if isinstance(v, (list, tuple)): return [_jsonable(x) for x in v]
    if isinstance(v, dict): return {k: _jsonable(x) for k, x in v.items()}
    return v

payload = {}
for _n, _r in results.items():
    rec = {k: _jsonable(v) for k, v in _r.items() if k not in ("G", "atoms")}
    st = structures.get(_n, {})
    for _f in ("pld", "lcd", "mofid", "url", "database"):
        if _f in st and _f not in rec:
            rec[_f] = _jsonable(st[_f])
    payload[_n] = rec

_p = OUT_DIR / "real_results.json"   # NOT results.json: that name belongs to the 77 hMOF run, and overwriting it
                                     # would destroy the set this one is meant to be compared against
_p.write_text(json.dumps(payload, indent=2))
print(f"SAVED {len(payload)} spectra -> {_p}  ({_p.stat().st_size/1024:.0f} KB)")
print()
print("Download it with:")
print("    from google.colab import files")
print(f"    files.download('{_p}')")


## Step 8 — Validation

Same four-part logic as before, adapted to a set that isn't guaranteed to contain a
hand-picked isostructural family:

1. **Duplicate/isomorphic check.** Any structures that happen to be graph-isomorphic (e.g.
   the database returning the same material processed twice) must return the same spectrum —
   checked automatically rather than assumed absent.
2. **Estimator comparison**, on one representative structure.
3. **Seed independence**, on the same structure.
4. **Finite-size scaling**, on the same structure.

In [ ]:
# ============================================================
# STEP 8: Validation
# ============================================================
# --- 1. automatic isomorphism check across the whole set ----------------------
print("Checking for accidental isomorphic duplicates in the fetched set...")
names_list = list(structures.keys())
iso_groups = []
seen = set()
for i, n1 in enumerate(names_list):
    if n1 in seen:
        continue
    group = [n1]
    for n2 in names_list[i+1:]:
        if n2 in seen:
            continue
        if structures[n1]["G_unit"].number_of_nodes() == structures[n2]["G_unit"].number_of_nodes() \
           and nx.is_isomorphic(structures[n1]["G_unit"], structures[n2]["G_unit"]):
            group.append(n2); seen.add(n2)
    if len(group) > 1:
        iso_groups.append(group)
    seen.add(n1)

if iso_groups:
    for g in iso_groups:
        widths = [results[n]["width"] for n in g if n in results]
        print(f"  isomorphic group {g}: dAlpha = {np.mean(widths):.3f} +/- {np.std(widths):.3f}")
else:
    print("  none found -- all fetched frameworks are topologically distinct unit-cell graphs.")

# --- 2. estimator comparison ---------------------------------------------------
probe = max(results, key=lambda n: results[n]["N"])   # largest successfully-run structure
G = structures[probe]["G"]
good = results[probe]
bad  = inmfa(G, average_pr_first=True)
print(f"\nEstimator comparison on {probe} (same graph, same coverings):")
print(f"  average ln P_q per realisation : dAlpha = {good['width']:.3f} +/- {good['width_sd']:.3f} "
      f"({100*good['width_sd']/good['width']:.0f}% scatter)")
print(f"  average pr first (old, wrong)  : dAlpha = {bad['width']:.3f} +/- {bad['width_sd']:.3f} "
      f"({100*bad['width_sd']/max(bad['width'],1e-9):.0f}% scatter)")

# --- 3. seed independence -------------------------------------------------------
print(f"\nSeed independence on {probe}:")
for sd in (0, 500, 1234):
    r = inmfa(G, seed=sd)
    print(f"  seed={sd:5d}  dAlpha={r['width']:.3f} +/- {r['width_sd']:.3f}  A={r['asymmetry']:+.2f}")

# --- 4. finite-size scaling ------------------------------------------------------
print(f"\nFinite-size scaling of Delta-alpha on {probe}:")
base_atoms = structures[probe]["atoms"]
for reps in [(1,1,1), (2,2,2)]:
    sc = base_atoms.repeat(reps)
    if len(sc) > MAX_ATOMS:
        print(f"  rep={reps}: skipped, {len(sc)} atoms exceeds MAX_ATOMS={MAX_ATOMS}")
        continue
    Gsc = build_graph(sc, pbc=True)
    if not nx.is_connected(Gsc):
        print(f"  rep={reps}: skipped, disconnected")
        continue
    r = inmfa(Gsc)
    print(f"  rep={reps}  N={len(sc):5d}  dAlpha={r['width']:.3f} +/- {r['width_sd']:.3f}")


## Step 9 — Combined spectra across all frameworks

In [ ]:
# ============================================================
# STEP 9: Combined multifractal spectrum plot
# ============================================================
order = sorted(results, key=lambda n: -results[n]["width"])
cmap = plt.cm.viridis(np.linspace(0, 1, len(order)))

fig, (ax, bx) = plt.subplots(1, 2, figsize=(17, max(6.5, 0.22 * len(order))),
                             gridspec_kw={"width_ratios": [1.4, 1]})

for i, name in enumerate(order):
    r = results[name]
    ax.errorbar(r["alpha"], r["f_alpha"], xerr=r["alpha_sd"], yerr=r["f_sd"],
                marker="o", ms=3, markevery=6, lw=1.0,
                elinewidth=0.4, capsize=0, errorevery=4, color=cmap[i], alpha=0.75)

ax.set_xlabel(r"$\alpha(q)$")
ax.set_ylabel(r"$f(\alpha)$")
ax.set_title(f"Multifractal spectra of {len(order)} real, Xe-tested CoRE MOF 2019 frameworks\n"
             f"color = rank by spectrum width (dark=narrow, yellow=wide); "
             f"{N_BATCHES}x{TRIALS_PER_BATCH} box coverings", fontsize=10)
ax.grid(alpha=0.25)

w  = [results[n]["width"] for n in order]
we = [results[n]["width_sd"] for n in order]
y  = np.arange(len(order))
bx.barh(y, w, xerr=we, color=cmap, alpha=0.9, capsize=1.5, height=0.8)
bx.set_yticks(y); bx.set_yticklabels(order, fontsize=6)
bx.invert_yaxis()
bx.set_xlabel(r"spectrum width $\Delta\alpha$")
bx.set_title("Structural heterogeneity ranking (all frameworks named)", fontsize=10)
bx.grid(alpha=0.25, axis="x")

plt.tight_layout()
out = OUT_DIR / "combined_multifractal_spectra_50plus.png"
plt.savefig(out, dpi=150)
plt.show()

payload = {n: {k: (v.tolist() if isinstance(v, np.ndarray) else v)
               for k, v in r.items()} for n, r in results.items()}
(OUT_DIR / "results.json").write_text(json.dumps(payload, indent=2))
print(f"\nSaved {len(results)} spectra to {OUT_DIR/'results.json'} and the combined plot to {out}")
print(f"\nWidth range across the set: {min(w):.3f} (narrowest: {order[-1]}) "
      f"to {max(w):.3f} (widest: {order[0]})")
print(f"Mean dAlpha = {np.mean(w):.3f}, std = {np.std(w):.3f}, "
      f"spread = {max(w)-min(w):.3f}")


## What the result says, and what it does not

**On the "all alike" worry, concretely:** with a hand-picked, non-random set of 8 (4 of them
a deliberate isostructural family), a tight cluster was ambiguous — it could mean the
frameworks really are similar, or it could mean the selection was too narrow. With 50+
independently-sourced real experimental structures selected by an objective, external
criterion (has a Xe isotherm on file in CoRE MOF 2019), the spread reported above (mean,
std, min/max) is the honest answer to "how alike are they really" — read it directly rather
than eyeballing the plot.

**Everything from the previous notebook's limitations section still applies, unchanged:**

- **Δα is finite-size dependent** (Step 8) — these are effective widths at a stated graph
  size, not asymptotic exponents.
- **This is topology only.** Bonds are unweighted edges; nothing here knows an element's
  polarisability or an open metal site's binding energy. Δα describes pore-network
  architecture, not Xe uptake, and must not be read as a selectivity predictor.
- **Nothing here is an adsorption calculation.** The "Xe relevance" of each framework is that
  MOFX-DB has *simulated* a Xe isotherm for it — that is the inclusion criterion, not
  something this notebook computes or verifies independently.
- **Bond perception is distance-cutoff based**, same caveat as before.
- **CoRE MOF 2019 structures are post-processed** (disordered/bound solvent handling per the
  CoRE MOF protocol) but are still derived from real, experimentally solved crystal
  structures — they are not hypothetical/generated frameworks.
- **New in this version:** the skip lists printed in Steps 3, 4, 5 and 7 are part of the
  result, not noise to ignore — they tell you exactly how many real candidates didn't survive
  automated parsing/bonding/analysis, and why.

## Step 9 — Is the spectrum the right SHAPE?

A multifractal spectrum `f(α)` must be an **inverted parabola** whose peak sits at `q = 0`,
where `f(α₀) = D₀`, the fractal dimension. If your curve just slopes downhill, something is
wrong with the mathematics, not with the material.

**This notebook previously had that bug.** The paper defines

$$\tau(q) = \frac{\ln \mathscr{P}_q(r)}{\ln(r/r_N)}$$

which is a straight-line fit **through the origin** — the relation `P_q ~ (r/r_N)^τ` has no
prefactor. The earlier code used `np.polyfit(x, y, 1)` and took the **slope**, which is a
fit with a **free intercept**. Those differ, and at `q = 0` the difference is fatal:

- at `q = 0` every term is `pr_i⁰ = 1`, so `P₀ = |I|`, the *count* of influential nodes
- that count is the **same at every radius**, so the free-intercept fit sees a flat line
- a flat line has slope 0, so `τ(0) = 0`, so `f(α₀) = 0·α − 0 = 0`

But `f(α₀)` is supposed to *be* the peak. Pinning it to zero deletes the peak and leaves a
falling curve. With the paper's definition `ln(r/r_N) < 0`, so `τ(0) = ln|I| / negative < 0`
and `f(α₀) = −τ(0) > 0` — the peak comes back.

`inmfa(..., tau_mode="paper")` is now the default. `tau_mode="slope"` reproduces the old
behaviour so the correction can be shown rather than asserted.


In [ ]:
# ============================================================
# STEP 9: Does the spectrum have the right SHAPE?
# ============================================================
# A multifractal spectrum f(alpha) must be an INVERTED PARABOLA whose peak sits
# at q = 0, where f(alpha_0) = D_0, the fractal dimension. This cell checks that
# directly, both ways, so the shape is verified rather than assumed.
import numpy as np

probe = sorted(results, key=lambda n: -results[n]["width"])[0]
G_probe = structures[probe]["G"]

print(f"probe framework: {probe}\n")
print(f"{'tau_mode':10s}{'tau(0)':>10s}{'f(q=0)':>10s}{'peak at q':>11s}"
      f"{'inverted parabola?':>21s}{'width':>8s}")
print("-" * 70)
shape = {}
for mode in ("slope", "paper"):
    r = inmfa(G_probe, tau_mode=mode, n_batches=3, trials_per_batch=4)
    q = np.asarray(r["q"]); f = np.asarray(r["f_alpha"]); tau = np.asarray(r["tau"])
    i0 = int(np.argmin(abs(q))); pk = int(np.nanargmax(f))
    shape[mode] = r
    print(f"{mode:10s}{tau[i0]:+10.4f}{f[i0]:+10.4f}{q[pk]:+11.1f}"
          f"{str(pk == i0):>21s}{r['width']:8.3f}")

print()
print("At q = 0 every term is pr_i^0 = 1, so P_0 = |I| -- the same at every radius.")
print("A flat line has zero slope, so the free-intercept fit forces tau(0) = 0 and")
print("therefore f(alpha_0) = 0. But f(alpha_0) is meant to be D_0, the PEAK. The")
print("paper's tau = ln P_q / ln(r/r_N) is a fit through the origin, which keeps")
print("tau(0) < 0 and restores the peak. Only 'paper' is the paper's mathematics.")


## Step 10 — The band, and a hard question about what it means

The band is the range of Δα that a group of frameworks shares, so a new framework can be
tested against it.

But before claiming the band means anything physical, it has to survive one check. Δα is
**finite-size dependent** — Step 7 shows it grows with the graph. This set has 77 frameworks
of very different sizes, which is finally enough to ask: *is Δα telling us about pore
architecture, or just about how big the supercell happened to be?*

The cell below answers that with partial correlations. Read its verdict before quoting any
pore-structure interpretation.


In [ ]:
# ============================================================
# STEP 10: The band, and whether Delta-alpha means what we hoped
# ============================================================
import numpy as np, json, os

def clean_curve(alpha, f_alpha):
    a = np.asarray(alpha, float); f = np.asarray(f_alpha, float)
    m = np.isfinite(a) & np.isfinite(f); a, f = a[m], f[m]
    o = np.argsort(a); a, f = a[o], f[o]
    au, inv = np.unique(a, return_inverse=True)
    return au, np.array([f[inv == k].mean() for k in range(len(au))])

W   = np.array([results[n]["width"] for n in results])
NAT = np.array([float(results[n]["N"]) for n in results])
PLD = np.array([float(structures[n].get("pld", np.nan)) for n in results])
LCD = np.array([float(structures[n].get("lcd", np.nan)) for n in results])

# ---- the band itself -------------------------------------------------------
lo, hi = np.percentile(W, [25, 75])
members = [n for n in results if lo <= results[n]["width"] <= hi]
print(f"Band from the interquartile range of Delta-alpha: {lo:.3f} - {hi:.3f}")
print(f"  {len(members)}/{len(results)} frameworks inside\n")

# ---- IS THE BAND MEASURING PORE ARCHITECTURE, OR JUST GRAPH SIZE? ----------
# This is the question that decides what the band is allowed to claim.
def pcorr(x, y, z):
    """partial correlation of x,y controlling for z"""
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x, y, z = x[ok], y[ok], z[ok]
    rxy = np.corrcoef(x, y)[0, 1]; rxz = np.corrcoef(x, z)[0, 1]; ryz = np.corrcoef(y, z)[0, 1]
    return (rxy - rxz * ryz) / np.sqrt((1 - rxz ** 2) * (1 - ryz ** 2))

def corr(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    return np.corrcoef(x[ok], y[ok])[0, 1]

print("Raw correlation of Delta-alpha with:")
for nm, v in [("pore limiting diam (PLD)", PLD), ("largest cavity diam (LCD)", LCD),
              ("number of atoms", NAT)]:
    print(f"   {nm:26s} r = {corr(W, v):+.3f}")

print("\nControlling for graph size (number of atoms):")
for nm, v in [("PLD", PLD), ("LCD", LCD)]:
    print(f"   Delta-alpha vs {nm:4s} | n_atoms   r = {pcorr(W, v, NAT):+.3f}")
print(f"   Delta-alpha vs size | LCD       r = {pcorr(W, NAT, LCD):+.3f}")

def r2(cols):
    ok = np.all(np.isfinite(np.column_stack(cols)), axis=1) & np.isfinite(W)
    X = np.column_stack([np.ones(ok.sum())] + [c[ok] for c in cols])
    b, *_ = np.linalg.lstsq(X, W[ok], rcond=None); p = X @ b
    return 1 - ((W[ok] - p) ** 2).sum() / ((W[ok] - W[ok].mean()) ** 2).sum()

print(f"\nVariance in Delta-alpha explained:")
print(f"   by graph size alone        R^2 = {r2([NAT]):.3f}")
print(f"   by pore size alone         R^2 = {r2([LCD]):.3f}")
print(f"   by both together           R^2 = {r2([NAT, LCD]):.3f}")
print(f"   -> pore size adds          {r2([NAT, LCD]) - r2([NAT]):+.3f}")

print("""
READ THIS BEFORE CLAIMING THE BAND MEANS PORE ARCHITECTURE.

The raw correlation with pore diameter looks supportive, but it does not
survive controlling for graph size: the partial correlation collapses to
roughly zero, while size survives controlling for pore diameter. Adding pore
size to a model that already has graph size buys almost no extra variance.

Delta-alpha is therefore tracking HOW BIG THE SUPERCELL IS, not how the pores
are arranged. And supercell size here is set by MIN_CELL_LENGTH and the
MAX_ATOMS cap -- computational parameters, not chemistry.

This is the finite-size dependence Step 7 warns about, now measured across the
whole set instead of three structures. Two ways forward, neither of them
'quote the raw correlation':

  1. Compare only frameworks at comparable graph size, so size is held roughly
     fixed and any residual grouping is structural.
  2. Normalise the fit window per structure so Delta-alpha stops growing with
     the graph, then re-test the pore correlation.
""")


## Step 11 — Figures

In [ ]:
# ============================================================
# STEP 11: Figures — the shape fix, the band, and the confound
# ============================================================
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(16, 9))
gs = fig.add_gridspec(2, 3, hspace=0.32, wspace=0.28)

# (a) the tau definition, and what it does to the shape
ax = fig.add_subplot(gs[0, 0])
for mode, c in [("slope", "#d13b3b"), ("paper", "#1f5fa8")]:
    r = shape[mode]; q = np.asarray(r["q"]); a = np.asarray(r["alpha"]); f = np.asarray(r["f_alpha"])
    ax.plot(a, f, "o-", ms=3, color=c,
            label=("free-intercept slope" if mode == "slope" else r"paper: $\ln P_q/\ln(r/r_N)$"))
    j = int(np.argmin(abs(q))); ax.plot(a[j], f[j], "*", ms=16, color=c, mec="k", mew=.7, zorder=5)
ax.set_xlabel(r"$\alpha$"); ax.set_ylabel(r"$f(\alpha)$")
ax.set_title(r"(a) $\tau$ definition fixes the shape" "\n★ = q=0, which must be the peak", fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=.25)

# (b) all spectra, correct definition
ax = fig.add_subplot(gs[0, 1])
for n in list(results)[:40]:
    a, f = clean_curve(results[n]["alpha"], results[n]["f_alpha"])
    ax.plot(a, f, lw=.8, alpha=.55)
ax.set_xlabel(r"$\alpha$"); ax.set_ylabel(r"$f(\alpha)$")
ax.set_title(f"(b) spectra, {min(40,len(results))} of {len(results)} frameworks", fontsize=10)
ax.grid(alpha=.25)

# (c) the Delta-alpha band
ax = fig.add_subplot(gs[0, 2])
order = sorted(results, key=lambda n: results[n]["width"])
ws = [results[n]["width"] for n in order]
ax.axhspan(lo, hi, color="#8a56c9", alpha=.16, label=f"band {lo:.2f}–{hi:.2f}")
ax.plot(range(len(ws)), ws, "o", ms=4, color="#333")
ax.set_xlabel("framework (sorted)"); ax.set_ylabel(r"$\Delta\alpha$")
ax.set_title("(c) the Δα band", fontsize=10); ax.legend(fontsize=8); ax.grid(alpha=.25)

# (d) the apparent pore correlation
ax = fig.add_subplot(gs[1, 0])
ok = np.isfinite(LCD) & np.isfinite(W)
ax.scatter(LCD[ok], W[ok], s=26, c=NAT[ok], cmap="viridis")
ax.set_xlabel("largest cavity diameter (Å)"); ax.set_ylabel(r"$\Delta\alpha$")
ax.set_title(f"(d) looks like pore size\n r = {corr(W,LCD):+.3f}  (colour = n_atoms)", fontsize=10)
ax.grid(alpha=.25)

# (e) but size explains it
ax = fig.add_subplot(gs[1, 1])
ax.scatter(NAT[ok], W[ok], s=26, c=LCD[ok], cmap="plasma")
ax.set_xlabel("number of atoms in supercell"); ax.set_ylabel(r"$\Delta\alpha$")
ax.set_title(f"(e) …but it is really graph size\n r = {corr(W,NAT):+.3f}  (colour = LCD)", fontsize=10)
ax.grid(alpha=.25)

# (f) the partial correlations, side by side
ax = fig.add_subplot(gs[1, 2])
labels = ["LCD\n(raw)", "LCD\n| size", "size\n(raw)", "size\n| LCD"]
vals = [corr(W, LCD), pcorr(W, LCD, NAT), corr(W, NAT), pcorr(W, NAT, LCD)]
cols = ["#2f7fd1", "#9fc4e8", "#d13b3b", "#efa4a4"]
ax.bar(labels, vals, color=cols)
ax.axhline(0, color="#333", lw=1)
for i, v in enumerate(vals):
    ax.text(i, v + (.02 if v >= 0 else -.05), f"{v:+.2f}", ha="center", fontsize=9, fontweight="bold")
ax.set_ylabel("correlation with Δα")
ax.set_title("(f) pore correlation vanishes\nwhen size is controlled for", fontsize=10)
ax.grid(alpha=.25, axis="y")

plt.savefig(OUT_DIR / "band77_analysis.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"saved {OUT_DIR / 'band77_analysis.png'}")


## What this notebook establishes — and what it retracts

**Fixed here.**

- **τ(q) now matches the paper.** The spectrum is a proper inverted parabola peaking at
  `q = 0`, and `α₀`, the asymmetry `A` and `D(q)` are meaningful for the first time. Every
  earlier number computed with the free-intercept slope was the wrong quantity.
- **α₀ is taken where `f(α)` is maximum**, per the paper, rather than assumed to be the
  `q = 0` index.

**Retracted.**

- **"Δα groups MOFs by pore architecture" is not supported by this data.** The raw
  correlation with pore diameter looks encouraging (r ≈ −0.33), but it **does not survive
  controlling for graph size**: the partial correlation collapses to ≈ +0.06, while graph
  size survives controlling for pore size (≈ +0.38). Adding pore diameter to a model that
  already contains graph size raises R² by about 0.002.
- What Δα mostly measures on this data is **the number of atoms in the supercell**, which is
  set by `MIN_CELL_LENGTH` and the `MAX_ATOMS` cap — computational parameters, not chemistry.

**What would make the claim testable.**

1. Compare only frameworks at **comparable graph size**, holding the confound roughly fixed.
2. **Normalise the fit window** per structure so Δα stops growing with the graph, then
   re-test the pore correlation.

Reporting the confound is the result here. A band built on an uncontrolled size effect would
not survive the first question from anyone who checked it.


---

## Step 12 — Compare against the 77-framework hypothetical band

This is the cell the whole notebook exists for. It puts the real frameworks and
the hypothetical ones on one axis and reports whether they occupy the same
region of Δα.

**Size is controlled first.** Δα rises with graph size, so a comparison between
sets of different sizes measures the size difference. Both sets are expanded
under the same `MIN_CELL_LENGTH` rule, and the cell reports the block-count
ranges so you can confirm they overlap before reading anything into the result.

To include the hMOF band, upload `results.json` from the original run next to
this notebook. Without it the cell still reports the real-MOF band on its own.


In [ ]:
# ============================================================
# STEP 12: Real vs hypothetical, on one axis
# ============================================================
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

real_w  = np.array([r["width"] for r in results.values() if np.isfinite(r.get("width", np.nan))])
real_bl = np.array([r.get("n_nodes", 0) + r.get("n_linkers", 0) for r in results.values()])

# The hMOF results are optional -- upload results.json to enable the comparison.
hmof_path = next((p for p in [Path("results.json"), Path("multi_mof_output/results.json_hmof"),
                              Path("hmof_results.json")] if p.exists()), None)
hmof_w = hmof_bl = None
if hmof_path:
    _h = json.loads(hmof_path.read_text())
    hmof_w  = np.array([v["width"] for v in _h.values() if np.isfinite(v.get("width", np.nan))])
    hmof_bl = np.array([v.get("n_nodes", 0) + v.get("n_linkers", 0) for v in _h.values()])
    print(f"loaded {len(hmof_w)} hypothetical frameworks from {hmof_path}")
else:
    print("results.json (the hMOF run) not found -- reporting the real band alone.")
    print("Upload it beside this notebook to get the comparison.")

def band(x):
    q1, q3 = np.percentile(x, [25, 75])
    return dict(n=len(x), lo=float(x.min()), hi=float(x.max()),
                q1=float(q1), q3=float(q3), med=float(np.median(x)))

rb = band(real_w)
print(f"\nREAL  (synthesised)   n={rb['n']:3d}  Δα {rb['lo']:.3f}–{rb['hi']:.3f}"
      f"  IQR {rb['q1']:.3f}–{rb['q3']:.3f}  median {rb['med']:.3f}")
print(f"      blocks {real_bl.min()}–{real_bl.max()}")

if hmof_w is not None:
    hb = band(hmof_w)
    print(f"hMOF  (hypothetical)  n={hb['n']:3d}  Δα {hb['lo']:.3f}–{hb['hi']:.3f}"
          f"  IQR {hb['q1']:.3f}–{hb['q3']:.3f}  median {hb['med']:.3f}")
    print(f"      blocks {hmof_bl.min()}–{hmof_bl.max()}")

    # --- is the comparison size-fair? ---
    overlap_lo = max(real_bl.min(), hmof_bl.min())
    overlap_hi = min(real_bl.max(), hmof_bl.max())
    print(f"\nblock-count overlap: {overlap_lo}–{overlap_hi}", end="  ")
    if overlap_hi <= overlap_lo:
        print("*** NO OVERLAP -- any Δα difference may be pure size. Do not claim a gap.")
    else:
        rm = (real_bl >= overlap_lo) & (real_bl <= overlap_hi)
        hm = (hmof_bl >= overlap_lo) & (hmof_bl <= overlap_hi)
        print(f"({rm.sum()} real, {hm.sum()} hypothetical in range)")
        if rm.sum() >= 3 and hm.sum() >= 3:
            print(f"  size-matched:  real Δα {real_w[rm].min():.3f}–{real_w[rm].max():.3f}"
                  f"   vs   hMOF {hmof_w[hm].min():.3f}–{hmof_w[hm].max():.3f}")
            gap = real_w[rm].min() - hmof_w[hm].max()
            print(f"  -> {'SEPARATED, gap %.3f' % gap if gap > 0 else 'OVERLAPPING (no gap)'}")

    # --- correlation of width with size, within each set ---
    for nm, w_, b_ in (("real", real_w, real_bl), ("hMOF", hmof_w, hmof_bl)):
        if len(w_) > 3 and b_.std() > 0:
            print(f"  r(Δα, blocks) within {nm}: {np.corrcoef(w_, b_)[0,1]:+.3f}")

# ------------------------------------------------------------------ figure --
fig, ax = plt.subplots(figsize=(9, 3.4), dpi=140)
sets = [("Synthesised (CoRE MOF)", real_w, "#b91c1c")]
if hmof_w is not None:
    sets.insert(0, ("Hypothetical (hMOF)", hmof_w, "#1d4ed8"))
for i, (label, vals, col) in enumerate(sets):
    y = len(sets) - 1 - i
    q1, q3 = np.percentile(vals, [25, 75])
    ax.plot([vals.min(), vals.max()], [y, y], color=col, lw=1.4, alpha=.55, zorder=1)
    ax.add_patch(plt.Rectangle((q1, y - .22), q3 - q1, .44,
                               facecolor=col, alpha=.16, edgecolor=col, lw=1.2, zorder=2))
    ax.plot([np.median(vals)] * 2, [y - .22, y + .22], color=col, lw=2.4, zorder=3)
    ax.scatter(vals, np.full_like(vals, y), s=13, color=col, alpha=.75, zorder=4)
    ax.text(-0.012, y, f"{label}\nn = {len(vals)}", transform=ax.get_yaxis_transform(),
            ha="right", va="center", fontsize=9)
ax.set_yticks([]); ax.set_ylim(-.6, len(sets) - .4)
ax.set_xlabel(r"$\Delta\alpha$   (multifractal spectrum width)")
ax.set_title("Spectrum width: synthesised vs hypothetical frameworks", fontsize=11)
ax.grid(axis="x", alpha=.25); ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / "real_vs_hypothetical.png", dpi=160, bbox_inches="tight")
plt.show()
print("\nsaved", OUT_DIR / "real_vs_hypothetical.png")


## Step 13 — Download everything

`real_results.json` is the file to send back. The figures go on the Results page
of the project site next to the 77-framework band.


In [ ]:
# ============================================================
# STEP 13: Download — and diagnose, if there is nothing to download
# ============================================================
# This cell used to assume the run had succeeded. When it had not, iterdir()
# threw FileNotFoundError with no output at all, which tells you nothing about
# WHICH step actually failed. It now reports the state of the run first.
from pathlib import Path
import shutil

OUT_DIR = Path("multi_mof_output")

print("=" * 62)
print("RUN STATE")
print("=" * 62)

# --- did each stage leave its variable behind? --------------------------
stages = [
    ("candidates", "Step 2  — records fetched from MOFX-DB"),
    ("structures", "Step 3  — CIFs parsed"),
    ("results",    "Step 7  — spectra computed"),
]
last_ok = None
for var, label in stages:
    obj = globals().get(var)
    if obj is None:
        print(f"  [ FAIL ] {label}: `{var}` does not exist — this step never ran")
        break
    print(f"  [  OK  ] {label}: {len(obj)} items")
    last_ok = var

if last_ok != "results":
    print()
    print("  The run stopped before the spectra were computed.")
    print("  Scroll UP to the first cell with a red error and read its message.")
    print("  The usual causes:")
    print("    * Step 2 assertion — too few verified CoRE MOF records came back.")
    print("      Fix: set TARGET_GAS = None in Step 1 and re-run Step 2, or lower")
    print("      MIN_MOFS, or put CIFs in real_cifs/ and set USE_LIVE_QUERY = False.")
    print("    * Step 3 assertion — too few CIFs parsed cleanly.")
    print("    * A stopped/restarted kernel — variables from earlier cells are gone.")
    print("      Fix: Runtime > Run all.")
    raise SystemExit("nothing to download yet — see above")

# --- what is actually on disk -------------------------------------------
print()
print("=" * 62)
print("FILES PRODUCED")
print("=" * 62)
if not OUT_DIR.exists():
    print(f"  {OUT_DIR}/ does not exist.")
    print("  The spectra were computed but never saved. Re-run Step 6b (the")
    print("  'SAVE IMMEDIATELY' cell), which is what writes real_results.json.")
    raise SystemExit("no output directory")

bundle = sorted(p for p in OUT_DIR.iterdir() if p.suffix in (".json", ".png"))
if not bundle:
    print(f"  {OUT_DIR}/ is empty. Re-run Step 6b to write real_results.json.")
    raise SystemExit("output directory is empty")

for p in bundle:
    print(f"  {p.name:38s} {p.stat().st_size/1024:8.1f} KB")

# --- the one file that matters ------------------------------------------
target = OUT_DIR / "real_results.json"
print()
if target.exists():
    import json as _json
    n = len(_json.loads(target.read_text()))
    print(f"  >>> real_results.json contains {n} frameworks. THIS is the file to send back.")
else:
    print("  *** real_results.json is MISSING. Re-run Step 6b.")
    print(f"      (found instead: {[p.name for p in bundle]})")

# --- bundle and download -------------------------------------------------
zip_path = shutil.make_archive("real_mof_results", "zip", OUT_DIR)
print(f"\nbundled -> {zip_path}  ({Path(zip_path).stat().st_size/1024:.0f} KB)")

try:
    from google.colab import files
    files.download(zip_path)
    print("\nIf no download started, your browser blocked it. Open the folder icon")
    print("in the left sidebar, right-click real_mof_results.zip, and Download.")
except ImportError:
    print("\nNot on Colab — copy multi_mof_output/ off this machine manually.")
except Exception as e:
    print(f"\nAutomatic download failed ({type(e).__name__}). Use the file browser:")
    print("  left sidebar > folder icon > real_mof_results.zip > right-click > Download")
